# Notebook 05 — Conversion Prediction Model
**Goal:** Build a classifier to predict which customers are most likely to subscribe.
Use SHAP to explain what drives conversion — enabling smarter targeting decisions.
Output: Scored customer list ranked by conversion probability.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import sys
sys.path.append('../src')
from model import (
    encode_features, train_models, evaluate,
    plot_roc_curves, plot_precision_recall,
    shap_analysis, score_customers
)
from eda_utils import save

df = pd.read_csv('../data/processed/segmented_data.csv')
print(f'Records: {len(df):,}')

## 1. Feature Encoding & Train/Test Split

In [ ]:
X, y = encode_features(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train conversion rate: {y_train.mean()*100:.2f}%')
print(f'Test  conversion rate: {y_test.mean()*100:.2f}%')
print(f'\nFeatures: {list(X.columns)}')

## 2. Train Models — Logistic Regression & XGBoost

In [ ]:
lr, xgb_model = train_models(X_train, y_train)
print('Models trained.')

## 3. Evaluation

In [ ]:
auc_lr, ap_lr, prob_lr = evaluate(lr, X_test, y_test, 'Logistic Regression')
auc_xgb, ap_xgb, prob_xgb = evaluate(xgb_model, X_test, y_test, 'XGBoost')

## 4. ROC & Precision-Recall Curves

In [ ]:
probs = {'Logistic Regression': prob_lr, 'XGBoost': prob_xgb}

fig = plot_roc_curves(y_test, probs)
save(fig, '16_roc_curves.png')
plt.show()

fig = plot_precision_recall(y_test, probs)
save(fig, '17_precision_recall.png')
plt.show()

## 5. SHAP Feature Importance (XGBoost)

In [ ]:
fig, shap_vals = shap_analysis(xgb_model, X_test, list(X.columns))
save(fig, '18_shap_importance.png')
plt.show()

## 6. SHAP Summary Plot

In [ ]:
import shap
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, feature_names=list(X.columns), show=False)
plt.title('SHAP Summary — Impact on Conversion Probability', fontweight='bold')
plt.tight_layout()
save(fig, '19_shap_summary.png')
plt.show()

## 7. Score All Customers & Save

In [ ]:
scored = score_customers(xgb_model, X, df)
scored.to_csv('../data/processed/scored_customers.csv', index=False)

print('Top 10 highest-probability customers:')
print(scored[['age','job','education','contact','campaign','conv_probability','priority_rank']].head(10))

## 8. Targeting Efficiency Analysis

In [ ]:
# If we only contact top 30% by probability, what % of conversions do we capture?
scored_sorted = scored.sort_values('conv_probability', ascending=False).reset_index(drop=True)
scored_sorted['cum_conv'] = scored_sorted['subscribed'].cumsum()
scored_sorted['pct_pop'] = (scored_sorted.index + 1) / len(scored_sorted) * 100
scored_sorted['pct_conv'] = scored_sorted['cum_conv'] / scored_sorted['subscribed'].sum() * 100

top30 = scored_sorted[scored_sorted['pct_pop'] <= 30]['pct_conv'].max()
print(f'Contacting top 30% by model score captures {top30:.1f}% of all conversions')

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(scored_sorted['pct_pop'], scored_sorted['pct_conv'], color='#1F4E79', linewidth=2.5, label='Model')
ax.plot([0,100],[0,100], 'k--', linewidth=1, label='Random')
ax.axvline(30, color='#F4A261', linestyle='--', linewidth=1.5, label='Top 30% cutoff')
ax.set_xlabel('% Population Contacted')
ax.set_ylabel('% Conversions Captured')
ax.set_title('Cumulative Gain Curve', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
save(fig, '20_cumulative_gain.png')
plt.show()